# Chapter 1: What is an Agent?

Imagine you want to keep up with new AI papers without reading the
whole arXiv feed every morning. So you write a small script: fetch
today's papers, pick the one that best matches your interest, and
summarize it. On a good day it works beautifully. You get one paper,
one summary, and your coffee is still warm.

Then comes a day where nothing new matches your interest. The script
doesn't know that. It still picks a paper, because picking a paper is
step two, and it still summarizes it, because summarizing is step
three. You get a confident summary of something you never wanted to
read, and nothing in the output tells you it went wrong. The script
did exactly what it was told. The problem is that it was told the same
thing no matter what it found.

:::{card} Chapter 1 of 11 · Foundations

**Where we left off:** Nowhere yet, this is the first chapter. Nothing to leave behind.

**What we fix now:** See why a script with fixed steps breaks when its results surprise it, and build the simplest possible program that looks at what it found before deciding what to do next.

**By the end, you can:**
- explain the difference between a fixed pipeline and an agent loop
- build a paper scout that observes, decides, and acts, step by step
- identify exactly why hand-written decision rules stop working as new situations show up
:::

```mermaid
flowchart TB
    subgraph F["Foundations"]
        direction LR
        LOOP[Agent loop] --> MOD[Model decides] --> TOOL[Tools and autonomy]
    end
    subgraph D["Continuous Development"]
        direction LR
        SCP[Scope and data] --> V1[Build v1] --> EVL[Evals and deploy]
    end
    subgraph C["Continuous Calibration, repeats"]
        direction LR
        RUN[Real use] --> ERR[Error patterns] --> FIX[Fixes] --> MET[New metrics]
    end
    F --> D --> C

    classDef active fill:#fef3c7,stroke:#d97706,stroke-width:3px,color:#78350f
    classDef done fill:#dbeafe,stroke:#93c5fd,stroke-width:1px,color:#1e3a8a
    classDef pending fill:#f3f4f6,stroke:#e5e7eb,stroke-width:1px,color:#9ca3af
    class LOOP active
    class MOD,TOOL,SCP,V1,EVL,RUN,ERR,FIX,MET pending
```

## The fix: look before you take the next step

A fixed pipeline decides everything in advance. Fetch, pick, summarize,
in that order, every time. An **agent** makes one change to that
picture: after each step, it looks at what happened and decides what
to do next. Three moves, repeated:

1. **Think**: look at everything gathered so far and choose the next step.
2. **Act**: do that one step, like running a search.
3. **Observe**: take the result back in, so the next "think" can use it.

The loop keeps going until the agent decides it's done, or until it
runs out of allowed steps. Here's the difference as a picture:

```mermaid
flowchart LR
    subgraph Pipeline["Fixed pipeline"]
        direction LR
        F1[Fetch] --> F2[Pick best] --> F3[Summarize]
    end
    subgraph Agent["Agent loop"]
        direction LR
        T[Think] --> A[Act] --> O[Observe] --> T
        T --> D[Done]
    end
```

Notice where the decision lives. In the pipeline, every decision was
made by you, before the script ever saw a single paper. In the loop,
some of the deciding happens *during* the run, after the program has
seen what's actually there. That's the whole idea behind the word
"agent": part of the control moves from the author to the program.

:::{tip} Try it

Start with the fixed version, so we have something to compare against.
The papers below are **made up for this chapter**, short enough to read
at a glance. Real arXiv papers come in later chapters.

The scout scores each paper by how often your interest keyword appears,
picks the highest score, and summarizes it by taking the first sentence
of the abstract. Run the cell, then change `interest` and run it again.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/velnmurugan/AgentTutorial/blob/main/chapters/01-what-is-an-agent.ipynb)

If the code below doesn't let you type into it, use the badge above to
open this same notebook in Google Colab, where you can edit and run it
freely.
:::

In [ ]:
import re

def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def score(paper, keyword):
    return words(paper["title"] + " " + paper["abstract"]).count(keyword)

def summarize(paper):
    return paper["abstract"].split(". ")[0] + "."

day_1 = [
    {"title": "Faster Retrieval for Long Documents",
     "abstract": "We propose a retrieval method that splits long documents into sections and searches them in parallel. Retrieval latency drops by half on three benchmarks."},
    {"title": "Teaching Robots to Fold Laundry",
     "abstract": "We train a robot arm to fold shirts from 200 demonstrations. It succeeds on 8 of 10 unseen garments."},
    {"title": "A Smaller Vision Transformer",
     "abstract": "We prune a vision transformer to a third of its size. Accuracy drops by less than one point."},
]

interest = "retrieval"

def fixed_scout(papers):
    best = max(papers, key=lambda p: score(p, interest))
    print("Picked:", best["title"], f"(score {score(best, interest)})")
    print("Summary:", summarize(best))

fixed_scout(day_1)

Three steps, always in the same order, and on day 1 that's all you
need. The best paper really is the best paper, so the summary is the
one you wanted.

:::{warning} Where it breaks

A fixed pipeline never checks whether its steps still make sense. Try
the same scout on a different day's papers (same notebook, same
[Colab link](https://colab.research.google.com/github/velnmurugan/AgentTutorial/blob/main/chapters/01-what-is-an-agent.ipynb) as before, if you need it):
:::

:::{tip} Predict before you run it

None of day 2's papers contain the word "retrieval", but one of them is
about RAG, which is exactly what you care about. Which paper do you
expect the fixed scout to summarize, and will anything in the output
warn you?
:::

In [ ]:
day_2 = [
    {"title": "Sparse Mixture-of-Experts at Scale",
     "abstract": "We route each token to 2 of 64 experts and train a 40B parameter model. Training cost falls by 30 percent."},
    {"title": "Protein Folding with Diffusion Models",
     "abstract": "We generate protein structures with a diffusion model. Predicted structures match lab results on 71 percent of targets."},
    {"title": "RAG Without Chunking",
     "abstract": "We show that RAG systems can skip chunking by indexing whole sections with a structure-aware encoder. Answer accuracy matches chunked baselines on two QA datasets."},
]

fixed_scout(day_2)

Every paper scores zero, so `max` just returns the first one in the
list. The scout summarizes a paper on mixture-of-experts, and the
output looks exactly as confident as it did on day 1. The only hint is
"score 0", and nothing in the script reacts to it.

Two things went wrong at once. The scout picked a paper you don't care
about, and it walked straight past the one you do, because "RAG" is a
different word from "retrieval". A person would have tried another
word, or said "nothing matched today". The script can't, because its
next step was decided before it saw the results.

## The same scout, as a loop

Here's the fix: split the scout into small actions, and put a
`decide` function in charge of choosing the next one after every step.
For now, `decide` is a plain list of rules we write by hand:

- if the last search found something, summarize the top result and stop
- otherwise, if there's a keyword we haven't tried yet, search for it
- otherwise, stop and say honestly that nothing matched

That's the whole agent. `decide` is the **think** step, running a
search or a summary is the **act** step, and storing the search result
in `state` is the **observe** step. `max_steps` is a safety limit, so
the loop can never run forever.

Run it on day 2, then try `day_1` and the empty `day_3`.

In [ ]:
keywords = ["retrieval", "rag", "search"]

def search(papers, keyword):
    hits = [p for p in papers if score(p, keyword) > 0]
    return sorted(hits, key=lambda p: score(p, keyword), reverse=True)

def decide(state):
    if state["found"]:
        return "summarize", state["found"][0]
    if state["tried"] < len(keywords):
        return "search", keywords[state["tried"]]
    return "stop", "Nothing relevant today."

def run_scout(papers, max_steps=5):
    state = {"tried": 0, "found": []}
    for step in range(1, max_steps + 1):
        action, arg = decide(state)                          # think
        if action == "search":
            state["found"] = search(papers, arg)             # act
            state["tried"] += 1                              # observe
            print(f"Step {step}: search '{arg}' -> {len(state['found'])} paper(s)")
        elif action == "summarize":
            print(f"Step {step}: summarize '{arg['title']}'")
            print("  Summary:", summarize(arg))
            return
        else:
            print(f"Step {step}: stop -> {arg}")
            return
    print("Out of steps.")

day_3 = [
    {"title": "Weather Forecasting with Graph Networks",
     "abstract": "We forecast 10 day weather on a global graph. Error falls by 12 percent against the previous best model."},
]

run_scout(day_2)

Read the printed steps top to bottom. The first search comes back
empty, and instead of pressing on, the loop tries the next keyword.
"rag" finds the right paper, so the next decision is to summarize it
and stop. On `day_3`, all three searches come back empty and the scout
says so, instead of summarizing a weather paper you never asked about.

Same papers, same summary function, different behavior. The only thing
that changed is *when* the decisions get made.

:::{warning} Where it breaks

The loop is only as smart as `decide`, and `decide` is a list of rules
we wrote before seeing any papers. Try it on day 4 (same
[Colab link](https://colab.research.google.com/github/velnmurugan/AgentTutorial/blob/main/chapters/01-what-is-an-agent.ipynb) as before):
:::

:::{tip} Predict before you run it

Day 4 has one paper that's genuinely about grounding answers in
documents (what you care about), and one paper about neural
architecture *search*, which has nothing to do with retrieval. Our
keyword list includes "search". Which paper do you expect the scout to
summarize?
:::

In [ ]:
day_4 = [
    {"title": "Neural Architecture Search for Tiny Devices",
     "abstract": "We search over 10,000 network designs to find models that fit on a microcontroller. The best design runs in 40 milliseconds."},
    {"title": "Grounding Answers in External Documents",
     "abstract": "We make a language model answer only from passages fetched from a document store. Unsupported claims fall by 60 percent."},
]

run_scout(day_4)

The scout summarizes the architecture search paper and never touches
the one you wanted. Both mistakes come from the same place: the rules
match *words*, and they don't understand *topics*. "Search" in
"architecture search" means something completely different from search
in "search your documents", and "grounding" and "fetched passages"
never appear in our keyword list at all.

You could patch it. Add "grounding" to the list, add a rule that skips
anything with "architecture" in the title. Then tomorrow brings a paper
about "evidence-backed generation", or a genuinely relevant paper that
happens to mention an architecture, and you're patching again. Every
new kind of paper needs a new rule, and you have to write each one
*before* the paper shows up. The loop is right; the thing doing the
deciding isn't flexible enough.

## Quick check

Before moving on, answer for yourself (no peeking ahead):

1. In your own words, what's the difference between the fixed scout and the loop scout? Where does the decision about the next step get made in each?

<details>
<summary>Check your reasoning ▸</summary>

A strong answer says the fixed scout's steps are all decided in advance by whoever wrote it, so it runs the same three steps no matter what it finds, while the loop scout decides its next step during the run, after looking at the result of the previous one. The key word is *when*: before seeing the papers, or after.

</details>

2. On day 2, the fixed scout printed "score 0" next to the paper it picked. Why didn't that stop it, and what would a sensible reaction to that score have been?

<details>
<summary>Check your reasoning ▸</summary>

A good answer points out that nothing in the fixed script reads the score after picking; summarizing is simply the next step. A sensible reaction would be to try a different keyword or to report that nothing matched, which is exactly what the loop version does once the decision can depend on what was observed.

</details>

3. In `run_scout`, which line is the "think" step, which is "act", and which is "observe"? What would happen if you deleted the "observe" line?

<details>
<summary>Check your reasoning ▸</summary>

`decide(state)` is think, `search(papers, arg)` is act, and updating `state` is observe. Without the observe step, `decide` would never see new results or know which keywords were tried, so it would make the same choice every time: the loop would run, but it would stop being able to change its mind.

</details>

4. Set `keywords = ["weather"]` and run the loop on `day_2`. Then imagine `decide` had a bug and always returned `"search"`. What does `max_steps` protect you from?

<details>
<summary>Check your reasoning ▸</summary>

With `["weather"]` on day 2, one empty search and the scout stops honestly. With an always-search bug, the loop would never reach a stop, and without `max_steps` it would run forever. A strong answer adds that in a real agent every step can cost time and money (a model call, an API request), so a step limit is a basic safety feature, not an afterthought.

</details>

5. Write one new rule for `decide` that fixes day 4. Then describe a paper, real or made up, that your new rule would get wrong.

<details>
<summary>Check your reasoning ▸</summary>

Any rule works as a starting point, like adding "grounding" to the keywords or skipping titles with "architecture". The important part is the second half: a good answer finds a paper that breaks the new rule, such as a relevant paper about a new *retrieval architecture*, or one that talks about "citing sources" without the word "grounding". That's the real lesson: hand-written rules can always be broken by the next paper nobody anticipated.

</details>

---

**You can now:** explain the difference between a fixed pipeline and an agent loop, and you've built a scout that decides its next step after seeing what it found.

**But one problem remains:** the decisions come from rules we wrote in advance, and they match words, not topics. Every new kind of paper needs a new rule. The next chapter hands the "think" step to a language model, which can judge whether a paper is relevant from what it actually says.

**Next → Chapter 2: Letting a Model Decide**